# Suite VAL — Validators

`Validate(registry)` checks data against schemas on demand: schema validity, exact native types, links set and
filled by the right schemas, and `unique(...)` clauses. Hand-written `Fake` visitables express data proxies never
produce.

In [ ]:
import math

from mbse_schemas.Framework import Proxies, Schemas as S, Validators
from support import Fake, raises, text

Person = S.OfObject.Builder().properties(text("name"), text("age", int)).create()
Pet = S.OfObject.Builder().properties(text("name")).create()
Rock = S.OfObject.Builder().properties(text("name")).create()  # declares no adjacency
Owns = S.OfRelation.Builder().links("owner", "item").properties(text("since"), text("share", float)).unique(
    "since", "share").create()  # (owner, item) determine the rest
Person = S.OfObject.Builder(Person).relations(lambda r: r.name("pets").of(Owns).me("owner")).update()
Pet = S.OfObject.Builder(Pet).relations(lambda r: r.name("owners").of(Owns).me("item")).update()
for name, schema in [("Person", Person), ("Pet", Pet), ("Rock", Rock), ("Owns", Owns)]:
    Proxies.register(name, schema)
B = Proxies.Builders
validate = Validators.Validate(B)

## VAL-01 · Natives

In [ ]:
assert validate(S.OfNative.Data(int), 3) == []
assert validate(S.OfNative.Data(int), True) == ["expected int, got bool"]
assert validate(S.OfNative.Data(float), 1) == ["expected float, got int"]
assert validate(S.OfNative.Data(list), []) == ["unsupported native type <class 'list'>"]
assert validate.OfNative(S.OfNative.Data(str), "x") == []
with raises(NotImplementedError):
    validate(S.OfUnion.Data(), 1)

## VAL-02 · Valid data has no problems

In [ ]:
ann = B.Person().name("Ann").age(30).create()
rex = B.Pet().name("Rex").create()
B.Person(ann).pets(lambda x: x.item(rex).since("2020")).update()
assert validate(Person, ann) == [] and validate.OfObject(Person, ann) == []
assert validate.Reachable(Person, ann) == [] and validate.Reachable(Pet, rex) == []
assert validate(Person, B.Person().create()) == []  # nothing is mandatory

## VAL-03 · Property types, with paths

In [ ]:
bad = B.Person().name(1).age(2.0).create()
assert validate(Person, bad) == ["Person#0.name: expected str, got int", "Person#0.age: expected int, got float"]
assert validate(Person, Fake("Person", {"colour": "red"})) == ["Person#0.colour: not a property of 'Person'"]

## VAL-04 · Entries: property types, unknown names, missing links, wrong schemas

In [ ]:
typed = B.Person().name("T").pets(lambda x: x.item(rex).since(2020)).create()
assert validate(Person, typed) == ["Person#0.pets[0].since: expected str, got int"]

fake_rex = Fake("Pet", {"name": "fake"})
cases = {
    "unknown adjacency": (Fake("Person", {}, {"friends": [{}]}), "Person#0.friends: not an adjacency of 'Person'"),
    "unknown property": (Fake("Person", {}, {"pets": [{"item": fake_rex, "colour": "red"}]}),
                         "Person#0.pets[0].colour: not a property of the relation"),
    "missing link": (Fake("Person", {}, {"pets": [{"since": "x"}]}), "Person#0.pets[0]: link 'item' is not set"),
    "wrong schema": (Fake("Person", {}, {"pets": [{"item": Fake("Rock", {"name": "r"})}]}),
                     "Person#0.pets[0].item: a 'Rock' cannot fill link 'item'"),
    "person as item": (Fake("Person", {}, {"pets": [{"item": Fake("Person")}]}),
                       "Person#0.pets[0].item: a 'Person' cannot fill link 'item'"),
}
for label, (obj, message) in cases.items():
    problems = validate(Person, obj)
    assert any(p.startswith(message) for p in problems), (label, problems)

## VAL-05 · Linked objects are checked when reached

In [ ]:
sick = B.Pet().name(0).create()
B.Person(ann).pets(lambda x: x.item(sick)).update()
assert validate(Person, ann) == []  # OfObject checks only the object and its entries
problems = validate.Reachable(Person, ann)
assert any("Pet#" in p and "name: expected str, got int" in p for p in problems)

## VAL-06 · `unique(...)`: agreeing on the rest means agreeing on the clause

In [ ]:
u1 = B.Person().name("U").create()
cat = B.Pet().name("Cat").create()
B.Person(u1).pets(lambda x: x.item(cat).since("a")).update()
assert validate(Person, u1) == []
B.Person(u1).pets(lambda x: x.item(cat).since("b")).update()
assert validate(Person, u1) == [
    "unique(share, since) violated: entries agreeing on ['item', 'owner'] differ on ['share', 'since']"]

## VAL-07 · Uniqueness uses schema equality: absent, -0.0, NaN, and int vs bool

In [ ]:
def owner_with(*entry_props):
    o = B.Person().name("E").create()
    pet = B.Pet().name("P").create()
    for props in entry_props:
        B.Person(o).pets(lambda x, props=props: [x.item(pet)] + [getattr(x, k)(v) for k, v in props.items()] and x).update()
    return o


assert validate(Person, owner_with({}, {"since": "a"}))  # absent vs present differ: a violation
assert validate(Person, owner_with({"share": 0.0}, {"share": -0.0}))  # 0.0 and -0.0 differ
assert validate(Person, owner_with({"share": math.nan}, {"share": math.nan})) == []  # NaNs are equal: one entry

## VAL-08 · Special clauses: over links only, empty, and over everything

In [ ]:
Pairs = S.OfRelation.Builder().links("l", "r").properties(text("k")).unique("r").create()  # l and k determine r
Empty = S.OfRelation.Builder().links("l", "r").properties(text("k")).unique().create()
All = S.OfRelation.Builder().links("l", "r").properties(text("k")).unique("l", "r", "k").create()
Box = S.OfObject.Builder().properties(text("n")).relations(
    lambda a: a.name("pairs").of(Pairs).me("l"), lambda a: a.name("pairs_r").of(Pairs).me("r"),
    lambda a: a.name("empty").of(Empty).me("l"), lambda a: a.name("empty_r").of(Empty).me("r"),
    lambda a: a.name("all").of(All).me("l"), lambda a: a.name("all_r").of(All).me("r")).create()
for name, schema in [("Box", Box), ("Pairs", Pairs), ("Empty", Empty), ("All", All)]:
    Proxies.register(name, schema)
box = B.Box().n("box").create()
b1, b2 = B.Box().n("1").create(), B.Box().n("2").create()
B.Box(box).pairs(lambda x: x.r(b1).k("same")).pairs(lambda x: x.r(b2).k("same")).update()
assert any(p.startswith("unique(r) violated") for p in validate(Box, box))
B.Box(box).empty(lambda x: x.r(b1).k("a")).empty(lambda x: x.r(b2).k("b")).update()
assert not any(p.startswith("unique() ") for p in validate(Box, box))  # unique() is trivially true
B.Box(box).all(lambda x: x.r(b1).k("a")).update()
assert not any("unique(k, l, r)" in p for p in validate(Box, box))
B.Box(box).all(lambda x: x.r(b2).k("b")).update()
assert any(p.startswith("unique(k, l, r) violated") for p in validate(Box, box))  # at most one entry

## VAL-09 · Schema problems are reported once per schema

In [ ]:
Broken = S.OfObject.Builder().properties(text("x", list)).create()
Proxies.register("Broken", Broken)
problems = validate.Reachable(Broken, B.Broken().create())
assert problems == ["schema 'Broken': property 'x': unsupported native type <class 'list'>"]

## VAL-10 · The root must be an instance of the schema given

In [ ]:
assert validate(Pet, ann) == ["the value is a 'Person', not an instance of the given schema"]

## VAL-11 · Finding F7 (pinned): uniqueness is checked only over the entries reached
`Tagged` says the tag alone determines the owner. Two components each use the tag "x"; validating one component
does not see the other, so the relation-wide violation goes unreported.

In [ ]:
Tagged = S.OfRelation.Builder().links("owner", "thing").properties(text("tag")).unique("owner", "thing").create()
Holder = S.OfObject.Builder().properties(text("n")).relations(lambda a: a.name("things").of(Tagged).me("owner")).create()
Thing = S.OfObject.Builder().properties(text("n")).relations(lambda a: a.name("holders").of(Tagged).me("thing")).create()
for name, schema in [("Tagged", Tagged), ("Holder", Holder), ("Thing", Thing)]:
    Proxies.register(name, schema)
h1 = B.Holder().n("h1").things(lambda x: x.thing(lambda t: t.n("t1")).tag("x")).create()
h2 = B.Holder().n("h2").things(lambda x: x.thing(lambda t: t.n("t2")).tag("x")).create()
assert validate.Reachable(Holder, h1) == [] and validate.Reachable(Holder, h2) == []
h3 = B.Holder().n("h3").things(lambda x: x.thing(lambda t: t.n("t3")).tag("y")).things(
    lambda x: x.thing(lambda t: t.n("t4")).tag("y")).create()
assert any(p.startswith("unique(owner, thing) violated") for p in validate(Holder, h3))  # within reach it is found

## VAL-12 · Validation reads only; it changes nothing

In [ ]:
from mbse_schemas.Framework import Plain

before = Plain.ToPlain.Reachable(Person, u1)  # u1 has a uniqueness violation but serializes fine
first = validate.Reachable(Person, u1)
assert validate.Reachable(Person, u1) == first  # repeatable
assert Plain.ToPlain.Reachable(Person, u1) == before
assert (ann.name, ann.age) == ("Ann", 30)

## VAL-13 · Property schemas that are not native cannot be validated yet

In [ ]:
Wrapper = S.OfObject.Builder().properties(lambda p: p.name("inner").of(Pet), text("label")).create()
Proxies.register("Wrapper", Wrapper)
assert validate(Wrapper, B.Wrapper().inner("x").label("ok").create()) == ["Wrapper#0.inner: only native values can be validated yet"]